# 05 — Feed-Forward Networks — Solution

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — Standard FFN

In [ ]:
class MyFFN(nn.Module):
    def __init__(self, d_model, d_ff=None, activation='gelu', dropout=0.0):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)
        self.act = {'relu': F.relu, 'gelu': F.gelu, 'silu': F.silu}[activation]

    def forward(self, x):
        return self.w2(self.drop(self.act(self.w1(x))))


ffn = MyFFN(64, 256, 'gelu')
x = torch.randn(2, 10, 64)
print('FFN output:', ffn(x).shape)

## Part B — Activation Functions

In [ ]:
x_vals = torch.linspace(-3, 3, 200)
activations = {'ReLU': F.relu(x_vals), 'GELU': F.gelu(x_vals), 'SiLU': F.silu(x_vals)}

plt.figure(figsize=(8, 4))
for name, y in activations.items():
    plt.plot(x_vals, y.detach(), label=name)
plt.axhline(0, color='k', linewidth=0.5); plt.axvline(0, color='k', linewidth=0.5)
plt.xlabel('x'); plt.ylabel('activation(x)')
plt.title('Activation Functions'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print('GELU and SiLU are smooth and slightly negative for x < 0 — better gradient flow than ReLU.')

## Part C — SwiGLU

In [ ]:
class MySwiGLU(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or int(2/3 * 4 * d_model)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)  # gate branch
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)  # value branch
        self.w_down = nn.Linear(d_ff, d_model, bias=False)  # output proj

    def forward(self, x):
        gate = F.silu(self.w_gate(x))   # gating signal
        up   = self.w_up(x)             # value signal
        return self.w_down(gate * up)   # element-wise gate, then project down


swiglu = MySwiGLU(64)
x = torch.randn(2, 10, 64)
print('SwiGLU output:', swiglu(x).shape)

ffn_params    = sum(p.numel() for p in MyFFN(64, 256).parameters())
swiglu_params = sum(p.numel() for p in MySwiGLU(64).parameters())
print(f'FFN params   : {ffn_params}')
print(f'SwiGLU params: {swiglu_params}  (should be comparable)')

## Part D — Library Comparison

In [ ]:
from src.models.feedforward import FeedForward, SwiGLU
x = torch.randn(2, 10, 64)
print('FeedForward:', FeedForward(64, 256, activation='gelu')(x).shape)
print('SwiGLU     :', SwiGLU(64)(x).shape)

# Intuition: why SwiGLU?
print('\nWhy SwiGLU?')
print('  - The gate learns WHICH information to pass through')
print('  - The up-projection learns WHAT information to pass')
print('  - Together they mimic a learned, dynamic activation function')
print('  - Empirically ~1-2% better perplexity vs GELU at same param count')